## 한국어 뉴스 주제 분류 파인튜닝

기본 모델: `klue/bert-base`  
데이터셋: `klue/ynat`  
결과: 한국어 뉴스 제목을 주제별로 분류

파인튜닝 전에는 기본 모델이 뉴스 주제 분류용으로 학습되어 있지 않기 때문에 원하는 라벨을 제대로 예측하지 못한다.  
파인튜닝 후에는 한국어 뉴스 제목을 입력했을 때 정치, 경제, 사회, 생활문화, 세계, IT과학, 스포츠 중 하나로 분류할 수 있다.

영화리뷰 감정분석
기본 모델: beomi/kcbert-base
데이터셋: NSMC
분류 라벨: 부정, 긍정
결과: 한국어 영화 리뷰의 감정을 분류

In [ ]:
!pip install -q -U "datasets<4.0.0" transformers evaluate accelerate

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 491.5/491.5 kB 32.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 9.0 MB/s eta 0:00:00


In [ ]:
import torch
import numpy as np
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer,
    pipeline
)
import evaluate

print("GPU 사용 가능 여부:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU 이름:", torch.cuda.get_device_name(0))

GPU 사용 가능 여부: True
GPU 이름: Tesla T4


0:정치 ,1:경제, 2:사회 , 3:문화생활,  4:세계, 5:IT과학, 6:스포츠


데이터 불러오기

In [ ]:
from datasets import load_dataset

dataset = load_dataset("klue/klue", "ynat", trust_remote_code=True)

dataset

`trust_remote_code` is not supported anymore.
Please check that the Hugging Face dataset 'klue/klue' isn't based on a loading script and remove `trust_remote_code`.
If the dataset is based on a loading script, please ask the dataset author to remove it and convert it to a standard format like Parquet.
ERROR:datasets.load:`trust_remote_code` is not supported anymore.
Please check that the Hugging Face dataset 'klue/klue' isn't based on a loading script and remove `trust_remote_code`.
If the dataset is based on a loading script, please ask the dataset author to remove it and convert it to a standard format like Parquet.


README.md:   0%|          | 0.00/22.5k [00:00<?, ?B/s]

ynat/train-00000-of-00001.parquet:   0%|          | 0.00/4.17M [00:00<?, ?B/s]

ynat/validation-00000-of-00001.parquet:   0%|          | 0.00/847k [00:00<?, ?B/s]

Generating train split:   0%|          | 0/45678 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/9107 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['guid', 'title', 'label', 'url', 'date'],
        num_rows: 45678
    })
    validation: Dataset({
        features: ['guid', 'title', 'label', 'url', 'date'],
        num_rows: 9107
    })
})

기본 모델과 토크나이저 불러오기

In [ ]:
for i in range(5):
    print("리뷰:", dataset["train"][i]["title"])
    print("라벨:", dataset["train"][i]["label"])
    print()

리뷰: 유튜브 내달 2일까지 크리에이터 지원 공간 운영
라벨: 3

리뷰: 어버이날 맑다가 흐려져…남부지방 옅은 황사
라벨: 3

리뷰: 내년부터 국가RD 평가 때 논문건수는 반영 않는다
라벨: 2

리뷰: 김명자 신임 과총 회장 원로와 젊은 과학자 지혜 모을 것
라벨: 2

리뷰: 회색인간 작가 김동식 양심고백 등 새 소설집 2권 출간
라벨: 3



In [62]:
train_dataset = dataset["train"].shuffle(seed=42).select(range(10000))
test_dataset = dataset["validation"].shuffle(seed=42).select(range(1500))

print(train_dataset)
print(test_dataset)

Dataset({
    features: ['guid', 'title', 'label', 'url', 'date'],
    num_rows: 10000
})
Dataset({
    features: ['guid', 'title', 'label', 'url', 'date'],
    num_rows: 1500
})


In [63]:
model_name = "klue/bert-base"

tokenizer = AutoTokenizer.from_pretrained(model_name)

In [64]:
model_before = AutoModelForSequenceClassification.from_pretrained(
    model_name,
    num_labels=7
)

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertForSequenceClassification LOAD REPORT from: klue/bert-base
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
classifier.bias                            | MISSING    | 
classifier.weight                          | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


In [65]:
def tokenize_function(examples):
    return tokenizer(
        examples["title"],
        padding="max_length",
        truncation=True,
        max_length=64
    )

tokenized_train = train_dataset.map(tokenize_function, batched=True)
tokenized_test = test_dataset.map(tokenize_function, batched=True)

Map:   0%|          | 0/10000 [00:00<?, ? examples/s]

Map:   0%|          | 0/1500 [00:00<?, ? examples/s]

In [66]:
print(tokenized_train.column_names)
print(tokenized_test.column_names)

['guid', 'title', 'label', 'url', 'date', 'input_ids', 'token_type_ids', 'attention_mask']
['guid', 'title', 'label', 'url', 'date', 'input_ids', 'token_type_ids', 'attention_mask']


In [67]:
# id 컬럼은 없을 수도 있으므로, 존재하는 컬럼만 제거
remove_cols_train = [col for col in ["guid","title","url","date"] if col in tokenized_train.column_names]
remove_cols_test = [col for col in ["guid","title","url","date"] if col in tokenized_test.column_names]

tokenized_train = tokenized_train.remove_columns(remove_cols_train)
tokenized_test = tokenized_test.remove_columns(remove_cols_test)

# set_format("torch")는 사용하지 않음
tokenized_train[0]

{'label': 3,
 'input_ids': [2,
  29104,
  3956,
  2223,
  2015,
  18556,
  16342,
  15146,
  100,
  1391,
  2437,
  7683,
  8189,
  3,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0],
 'token_type_ids': [0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0],
 'attention_mask': [1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  

In [68]:
model = AutoModelForSequenceClassification.from_pretrained(
    model_name,
    num_labels=7
)

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertForSequenceClassification LOAD REPORT from: klue/bert-base
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
classifier.bias                            | MISSING    | 
classifier.weight                          | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


In [69]:
accuracy = evaluate.load("accuracy")

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=-1)
    return accuracy.compute(predictions=predictions, references=labels)

In [70]:
training_args = TrainingArguments(
    output_dir="./nsmc_klue_bert_model",
    eval_strategy="epoch",
    save_strategy="epoch",
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    num_train_epochs=3,
    weight_decay=0.01,
    logging_steps=20,
    report_to="none",
    load_best_model_at_end=True
)

In [71]:
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_train,
    eval_dataset=tokenized_test,
    processing_class=tokenizer,
    compute_metrics=compute_metrics
)

1회 학습

3000개 -> 0.806 ( 테스트 500개 )

5000개 -> 0.8 ( 테스트 1000개)

10000개 -> 0.798 ( 테스트 1000개 )

20000개 -> 0.808 ( 테스트 1000개)

40000개 -> 0.792 ( 테스트 2000개)

3회 10000개 -> 0.879


In [72]:
trainer.train()

Epoch,Training Loss,Validation Loss,Accuracy
1,0.402884,0.404447,0.857333
2,0.266730,0.381629,0.878667
3,0.205176,0.401814,0.879333


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=1875, training_loss=0.3325091271718343, metrics={'train_runtime': 446.8375, 'train_samples_per_second': 67.139, 'train_steps_per_second': 4.196, 'total_flos': 986710752000000.0, 'train_loss': 0.3325091271718343, 'epoch': 3.0})

In [73]:
trainer.evaluate()

Training Loss,Validation Loss,Epoch,Accuracy
0.205176,0.381629,3,0.878667


{'eval_loss': 0.38162851333618164, 'eval_accuracy': 0.8786666666666667}

In [ ]:
trainer.save_model("./my_korean_sentiment_model")
tokenizer.save_pretrained("./my_korean_sentiment_model")

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

('./my_korean_sentiment_model/tokenizer_config.json',
 './my_korean_sentiment_model/tokenizer.json')

In [ ]:
after_classifier = pipeline(
    "text-classification",
    model="./my_korean_sentiment_model",
    tokenizer="./my_korean_sentiment_model",
    device=0 if torch.cuda.is_available() else -1
)
test_texts = [
    "유튜브 내달 2일까지 크리에이터 지원 공간 운영.",
    "어버이날 맑다가 흐려져…남부지방 옅은 황사.",
    "내년부터 국가RD 평가 때 논문건수는 반영 않는다.",
    "회색인간 작가 김동식 양심고백 등 새 소설집 2권 출간."
]

for text in test_texts:
    result = after_classifier(text)
    print("입력:", text)
    print("파인튜닝 후 결과:", result)
    print()

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

입력: 유튜브 내달 2일까지 크리에이터 지원 공간 운영.
파인튜닝 후 결과: [{'label': 'LABEL_3', 'score': 0.3260222375392914}]

입력: 어버이날 맑다가 흐려져…남부지방 옅은 황사.
파인튜닝 후 결과: [{'label': 'LABEL_3', 'score': 0.6994224190711975}]

입력: 내년부터 국가RD 평가 때 논문건수는 반영 않는다.
파인튜닝 후 결과: [{'label': 'LABEL_1', 'score': 0.29850658774375916}]

입력: 회색인간 작가 김동식 양심고백 등 새 소설집 2권 출간.
파인튜닝 후 결과: [{'label': 'LABEL_3', 'score': 0.7131615877151489}]



In [54]:
label_map = {
    "LABEL_0": "정치",
    "LABEL_1": "경제",
    "LABEL_2": "사회",
    "LABEL_3": "문화생활",
    "LABEL_4": "세계",
    "LABEL_5": "IT과학",
    "LABEL_6": "스포츠"
}

def predict_sentiment(text):
    result = after_classifier(text)[0]
    label = label_map[result["label"]]
    score = result["score"]

    print("입력 문장:", text)
    print("예측 결과:", label)
    print("확신도:", round(score, 4))

predict_sentiment("유튜브 내달 2일까지 크리에이터 지원 공간 운영.")
predict_sentiment("어버이날 맑다가 흐려져…남부지방 옅은 황사.")
predict_sentiment("내년부터 국가RD 평가 때 논문건수는 반영 않는다.")
predict_sentiment("회색인간 작가 김동식 양심고백 등 새 소설집 2권 출간.")

입력 문장: 유튜브 내달 2일까지 크리에이터 지원 공간 운영.
예측 결과: 문화생활
확신도: 0.326
입력 문장: 어버이날 맑다가 흐려져…남부지방 옅은 황사.
예측 결과: 문화생활
확신도: 0.6994
입력 문장: 내년부터 국가RD 평가 때 논문건수는 반영 않는다.
예측 결과: 경제
확신도: 0.2985
입력 문장: 회색인간 작가 김동식 양심고백 등 새 소설집 2권 출간.
예측 결과: 문화생활
확신도: 0.7132


In [56]:
# 기존 검증용(0~500번)과 겹치지 않게 500번 이후의 데이터에서 5개를 가져옵니다.
unseen_news = dataset["validation"].shuffle(seed=42).select(range(510,515))

# 모델이 원본 텍스트를 담고 있는 'title' 컬럼과 정답 라벨 'label' 컬럼을 추출합니다.
test_titles = unseen_news["title"]
test_labels = unseen_news["label"]

# YNAT 공식 대주제 라벨 이름 매핑 리스트
ynat_labels = ["정치", "경제", "사회", "문화생활", "세계", "IT과학", "스포츠"]

print("--- [YNAT] 미학습 데이터 추론 테스트 ---")
for text, true_label in zip(test_titles, test_labels):
    result = after_classifier(text)[0]

    print("파인튜닝 결과")
    print(predict_sentiment(text))
    print("실제 정답:", ynat_labels[true_label]) # 모델이 맞혔는지 확인할 수 있는 정답 출력
    print()

--- [YNAT] 미학습 데이터 추론 테스트 ---
파인튜닝 결과
입력 문장: 英 새 하원의장에 노동당 출신 린지 호일 경 선출
예측 결과: 세계
확신도: 0.7344
None
실제 정답: 세계

파인튜닝 결과
입력 문장: 질의하는 정경희 의원
예측 결과: 스포츠
확신도: 0.4855
None
실제 정답: 스포츠

파인튜닝 결과
입력 문장: 컨버터블 노트북 인기…삼성 펜 11만대 판매·LG 내년 가세
예측 결과: 정치
확신도: 0.6125
None
실제 정답: 정치

파인튜닝 결과
입력 문장: 정부 코로나19 자가격리자에 등기 발송…집배원 감염 위험종합
예측 결과: 사회
확신도: 0.3166
None
실제 정답: 사회

파인튜닝 결과
입력 문장: 올해 홍진기 창조인상에 여자 컬링 대표팀
예측 결과: IT과학
확신도: 0.771
None
실제 정답: IT과학

